##Testing the Frequency Branch on a StyleGAN generated dataset

In [ ]:
!pip install -q kagglehub

import kagglehub

path = kagglehub.dataset_download("xhlulu/140k-real-and-fake-faces")
print(path)

Using Colab cache for faster access to the '140k-real-and-fake-faces' dataset.
/kaggle/input/140k-real-and-fake-faces


##Download and Setup

In [ ]:
import os

DATASET_PATH = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"

print(DATASET_PATH)
print(os.listdir(DATASET_PATH))

/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train
['fake', 'real']


##Dataset Preparation

Selecting balanced subset , Applying the same
70/15/15 train/validation/test split and new augmentation method

In [ ]:
import os
import random
import torch

from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.datasets import ImageFolder



DATASET_PATH = "/kaggle/input/140k-real-and-fake-faces/real_vs_fake/real-vs-fake/train"


train_transform = transforms.Compose([   #og transforms
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5, 0.5, 0.5],
        std=[0.5, 0.5, 0.5]
    )
])

##loading dataset

full_dataset = ImageFolder(DATASET_PATH, transform=None)

print("Classes:", full_dataset.classes)
print("Class mapping:", full_dataset.class_to_idx)
print("Total available:", len(full_dataset))



random.seed(42)
torch.manual_seed(42)   #reproducibility

real_label = full_dataset.class_to_idx["real"]
fake_label = full_dataset.class_to_idx["fake"]

real_indices = [
    i for i, (_, label) in enumerate(full_dataset.samples)
    if label == real_label
]

fake_indices = [
    i for i, (_, label) in enumerate(full_dataset.samples)
    if label == fake_label
]

random.shuffle(real_indices)
random.shuffle(fake_indices)

selected_indices = real_indices[:8000] + fake_indices[:8000]
random.shuffle(selected_indices)

full_dataset.samples = [
    full_dataset.samples[i]
    for i in selected_indices
]

full_dataset.imgs = full_dataset.samples
full_dataset.targets = [
    label for _, label in full_dataset.samples
]

print("\nSelected dataset:")
print("Total:", len(full_dataset))
print(
    "Real:",
    sum(label == real_label for _, label in full_dataset.samples)
)
print(
    "Fake:",
    sum(label == fake_label for _, label in full_dataset.samples)
)

#same split throughout

train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

generator = torch.Generator().manual_seed(42)

train_sub, val_sub, test_sub = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=generator
)



class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        path, label = self.subset.dataset.samples[
            self.subset.indices[idx]
        ]

        image = self.subset.dataset.loader(path)

        return self.transform(image), label



train_set = TransformSubset(train_sub, train_transform)
val_set = TransformSubset(val_sub, test_transform)
test_set = TransformSubset(test_sub, test_transform)



train_loader = DataLoader(
    train_set,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_set,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

test_loader = DataLoader(
    test_set,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

print(
    f"\nTrain: {len(train_set)}, "
    f"Val: {len(val_set)}, "
    f"Test: {len(test_set)}"
)

Classes: ['fake', 'real']
Class mapping: {'fake': 0, 'real': 1}
Total available: 100000

Selected dataset:
Total: 16000
Real: 8000
Fake: 8000

Train: 11200, Val: 2400, Test: 2400


##Defining the refined Frequency Branch

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader, random_split , Dataset
import timm
import numpy as np
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt

class FrequencyBranch(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(6, 16, kernel_size=3, padding=1),  # 6 channels: RGB magnitude + RGB phase
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((4, 4))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

    def forward(self, x):

        # FFT now separately on R,G and B channels
        fft = torch.fft.fft2(x)
        fft = torch.fft.fftshift(fft, dim=(-2, -1))

        # Magnitude for each RGB channel
        magnitude = torch.abs(fft)
        magnitude = torch.log1p(magnitude)

        magnitude = (
            magnitude -
            magnitude.mean(dim=(-2, -1), keepdim=True)
        ) / (
            magnitude.std(dim=(-2, -1), keepdim=True) + 1e-6
        )

        # Phase for each RGB channel
        phase = torch.angle(fft)
        phase = phase / torch.pi

        # 3 magnitude + 3 phase = 6 input channels
        freq_input = torch.cat(
            [magnitude, phase],
            dim=1
        )

        out = self.features(freq_input)
        out = self.classifier(out)

        return out

class FrequencyOnlyDetector(nn.Module):  #standalone classifier-no spatial stream
    def __init__(self):
        super().__init__()
        self.freq_branch = FrequencyBranch()
        self.classifier = nn.Linear(128, 2)

    def forward(self, x):
        features = self.freq_branch(x)
        return self.classifier(features)

model = FrequencyOnlyDetector().cuda()
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

Trainable parameters: 155,698


##Optimizer and scheduler

In [ ]:
num_epochs = 15

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

##Training and validation functions

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.cuda(), labels.cuda()
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total

def val_epoch(model, loader, criterion):
    model.eval()
    running_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.cuda(), labels.cuda()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    return running_loss/len(loader), 100.*correct/total


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Training loop

To see if the model can learn

In [ ]:
best_val_acc = 0
train_losses, val_losses = [], []
train_accs, val_accs = [], []

for epoch in range(num_epochs):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_acc = val_epoch(model, val_loader, criterion)
    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_model_styleganrgb2.pth')
        print(f"  -> Best model saved")

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

Epoch 1/15 | Train Loss: 0.6943 | Train Acc: 50.68% | Val Loss: 0.6884 | Val Acc: 57.12%
  -> Best model saved
Epoch 2/15 | Train Loss: 0.6873 | Train Acc: 54.41% | Val Loss: 0.6786 | Val Acc: 58.42%
  -> Best model saved
Epoch 3/15 | Train Loss: 0.6676 | Train Acc: 60.47% | Val Loss: 0.6537 | Val Acc: 62.88%
  -> Best model saved
Epoch 4/15 | Train Loss: 0.6501 | Train Acc: 61.95% | Val Loss: 0.6482 | Val Acc: 62.50%
Epoch 5/15 | Train Loss: 0.6407 | Train Acc: 63.52% | Val Loss: 0.6458 | Val Acc: 62.33%
Epoch 6/15 | Train Loss: 0.6288 | Train Acc: 64.57% | Val Loss: 0.6286 | Val Acc: 65.25%
  -> Best model saved
Epoch 7/15 | Train Loss: 0.6239 | Train Acc: 65.62% | Val Loss: 0.6620 | Val Acc: 61.67%
Epoch 8/15 | Train Loss: 0.6195 | Train Acc: 66.09% | Val Loss: 0.6192 | Val Acc: 65.67%
  -> Best model saved
Epoch 9/15 | Train Loss: 0.6137 | Train Acc: 66.85% | Val Loss: 0.6271 | Val Acc: 65.25%
Epoch 10/15 | Train Loss: 0.6123 | Train Acc: 66.85% | Val Loss: 0.6194 | Val Acc: 66.42%